[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/05b_marl.ipynb)

# 05b — Multi-Agent Reinforcement Learning

**Purpose.** Learn a multi-agent policy over the absolute-tilt space with TorchRL
and produce `optimized_tilt_MARL`. PROJECT.md section 14.

**The comparison discipline.** This notebook and 05a solve the *same* problem:
the same bounds from `src.optim.space`, the same KPIs from `src.kpi`, the same
frozen surrogate (PROJECT.md section 17). What differs is how the space is
searched, and that difference is the result.

**The action is the tilt, not a change to it.** Each agent emits absolute tilts
in `[tilt_min, tilt_max]`, so the action space is fixed rather than moving with
the current configuration (PROJECT.md Decision 1 and section 14.3).

**The surrogate predicts a radio map.** `src/kpi/` derives the five KPIs from
`R_hat` exactly as it does from a ray-traced map (Decision 6), so the reward
here and the validation in section 8 differ only in which map they score.

**Requires** `uv sync --extra marl`.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("torchrl", "torchrl"), ("tensordict", "tensordict")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/external/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=["optim=marl", *CONFIG_OVERRIDES])
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Search space, surrogate, objective

Identical construction to notebook 05a. If these three cells ever differ between
the two notebooks, the comparison is invalid.

In [ ]:
from hydra.utils import instantiate

from src.data.load import load_cell_config
from src.optim.objective import Objective
from src.optim.space import TiltSpace
from src.radio import cell_band

table = cell_band.build_table(load_cell_config(cfg), cfg)
space = TiltSpace(table, cfg)

surrogate = instantiate(cfg.surrogate).load(cfg.surrogate.artifact_path)
objective = Objective(space, cfg, surrogate=surrogate)

tilt_0 = space.baseline()
baseline_kpis = objective.kpis(tilt_0, source="sionna")
pd.Series(baseline_kpis).to_frame("baseline").loc[list(cfg.kpi.order)]

## 3. Build the environment

The agent partition must cover every cell-band exactly once. A cell-band owned by
two agents has its tilt overwritten nondeterministically; one owned by none
silently keeps its baseline tilt and is quietly excluded from the optimization.

The environment scores through the **frozen surrogate**. Training against
Sionna-RT directly would take weeks (PROJECT.md section 11.4), and the surrogate
is not retrained on what the policy visits — freezing it is what keeps a
Sionna-RT solve out of the loop.

In [ ]:
from src.optim.marl.env import TiltEnv

env = TiltEnv(space, objective, cfg)
print(f"granularity: {cfg.optim.agents.granularity}")
print(f"reward source: {cfg.optim.env.reward_source}")

## 4. Sanity-check the environment before training

Cheap now, expensive later. Three things worth confirming with a few random
steps:

1. Every joint action assembles into an in-bounds tilt — a squashing bug shows
   up here rather than after hours of training.
2. The reward responds to the action at all. A constant reward means the agents
   are learning nothing regardless of how the curves look.
3. Assembling the joint tilt follows **cell-band table order**, not agent order.
   Getting that wrong assigns tilts to the wrong cells and produces a complete,
   plausible, entirely wrong training run.

In [ ]:
from src.radio import sampling

td = env.reset()
rewards = []
for _ in range(8):
    td = env.step(env.rand_action(td))
    rewards.append(float(td["next", "reward"].mean()))

print("rewards over 8 random steps:", np.round(rewards, 4))
assert np.std(rewards) > 0, "reward does not respond to the action"
sampling.assert_within_bounds(td["tilt"].numpy(), table)

## 5. Check the reward weights

`lambda_H > lambda_O > lambda_ON > lambda_BPS > lambda_W`, applied to
**normalised** KPIs. Raw-scale weighting lets the KPI scales set the effective
priority regardless of what the weights say — the exact failure PROJECT.md
section 25.3 warns about.

Note the ordering changed with the PROJECT.md rewrite: weak rate is now last,
below Band Priority Score, so a policy may trade weak coverage for band
coordination. Weights carried over from the previous ordering will quietly
optimise the wrong thing.

A weighted sum is in any case a lossy encoding of a lexicographic priority.
PROJECT.md section 14.5 prefers a hierarchical or constrained reward where
strict priority is required; if a scalar reward is used, say so when reporting.

The trainer asserts the ordering at construction; it is checked here too because
a config error found before a training run is free.

In [ ]:
from src.optim.marl import reward

reward.assert_weight_ordering(cfg)
pd.Series(dict(cfg.optim.reward.weights)).to_frame("lambda")

## 6. Train

Centralised training with decentralised execution: the critic sees the joint
state, each policy sees only its own observation. That is what makes a shared,
non-decomposable reward learnable — every agent's tilt affects every KPI, so a
decentralised critic cannot attribute a change in hole rate to one cell among
26.

In [ ]:
from src.utils import tracking

with tracking.start_run(cfg, run_name="marl"):
    tracking.log_config(cfg)
    trainer = instantiate(cfg.optim, env=env, cfg=cfg)
    history = trainer.train()

print(f"wall-clock: {history['wall_clock_s']:.0f}s over {history['frames']:,} frames")

## 7. Learning curves — reward *and* KPIs

Plot both. The reward is a weighted scalar and can improve while hole rate gets
worse; a reward curve alone cannot show that, and it is the failure mode a
scalarized encoding of a lexicographic priority is most prone to.

In [ ]:
curves = pd.DataFrame(history["log"])
fig, axes = plt.subplots(1, 1 + len(cfg.kpi.order), figsize=(18, 3))
axes[0].plot(curves["reward"], lw=1)
axes[0].set_title("reward", fontsize=9)
for ax, col in zip(axes[1:], cfg.kpi.order, strict=True):
    ax.plot(curves[col], lw=1)
    ax.axhline(baseline_kpis[col], ls="--", lw=1, color="grey")
    ax.set_title(col, fontsize=9)
plt.tight_layout()

## 8. Extract the configuration and validate it

Act greedily — the policy mean, not a sample. The deliverable is one
configuration to deploy, and a stochastic action makes the reported result
irreproducible for no benefit.

Then validate with Sionna-RT. A converged reward curve says the agent learned to
maximise the *surrogate*; what matters is the KPI vector of the configuration it
produces (PROJECT.md section 16 Phase 7).

In [ ]:
from src.evaluation import validate

optimized_tilt = trainer.extract_policy_configuration()
predicted = objective.kpis(optimized_tilt, source="surrogate")
validated = validate.validate(optimized_tilt, cfg, predicted=predicted)

pd.DataFrame(
    {
        "baseline": baseline_kpis,
        "surrogate prediction": validated["prediction"],
        "Sionna-RT ground truth": validated["ground_truth"],
        "gap": validated["gap"],
    }
).loc[list(cfg.kpi.order)]

## 9. Result

The tilt table is the deliverable — one row per cell-band, with `delta_tilt`
derived purely for reporting (PROJECT.md section 19 and Decision 1).

In [ ]:
from src.evaluation import report

tilts = report.tilt_table(optimized_tilt, table)
record = report.summary(validated, tilts, cfg)
report.export({"marl": record}, "reports/results")
tilts.head(15)

## 10. Handoff checklist

- [ ] The agent partition covers every cell-band exactly once.
- [ ] Reward weights satisfy `lambda_H > lambda_O > lambda_ON > lambda_BPS >
      lambda_W` on normalised KPIs — the post-rewrite ordering, not the old one.
- [ ] KPI curves were inspected, not only the reward curve.
- [ ] The extracted configuration is in bounds and was taken greedily.
- [ ] The reported KPIs come from Sionna-RT.
- [ ] Training cost is recorded — it is part of this method's cost, and whether
      it amortises depends on policy transfer being demonstrated, not assumed.
- [ ] The evaluation budget is comparable to TuRBO's in 05a (PROJECT.md
      section 17). Two methods given different budgets is not a comparison.
- [ ] The run covered every seed in `cfg.optim.seeds`.